# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 07 · Turning, braking and body-orientation features

**Observed-only hypotheses · 60 candidates · three separately tested families**

A player can curve away from the constant-velocity path, brake instead of overshooting, or orient differently from the current travel direction. These are physical hypotheses—not known future behavior.

[Step-and-turn movement research](https://arxiv.org/html/2603.17866v1) motivates separating displacement magnitude and turning. It studies a different football task and does not establish our expected RMSE gain. See `EXPERIMENT_PROTOCOL.md` for exact formulas, information-time assumptions and overlap with existing features.

No earlier-origin augmentation, model sweep or ensemble. This notebook creates new features for the **same 256 plays**, reuses the old arrival controls, and makes at most **nine new small fits**. Run cells in order and stop on an exception.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import pandas as pd
import plotly.io as pio

KIT = Path('/home/sagemaker-user/nfl_feature_round2')
OUT = Path('/home/sagemaker-user/nfl-feature-round2-results')
if not KIT.is_dir():
    raise FileNotFoundError('Upload and extract nfl_feature_round2.zip into /home/sagemaker-user first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
HTML = OUT / 'visualizations'

def run_stage(command, seconds):
    subprocess.run([sys.executable, str(KIT / 'run_round.py'), command,
                    '--out', str(OUT), '--seconds', str(seconds)], cwd=KIT, check=True)

def read_result(relative):
    path = OUT / relative
    if not path.is_file():
        raise FileNotFoundError(f'Missing stage result: {path}. Run the preceding stage; do not invent results.')
    return json.loads(path.read_text())

print('Kernel:', sys.executable)
print('Outputs:', OUT)
print('No new cloud jobs, package installation, Git writes or Kaggle submissions.')

assert read_result('attribution/summary.json')['status'] == 'attribution_complete', 'Finish notebook 06 first.'


## 1 · Test 32 training-side plays first
180-second cap. Only observed input CSVs are read. Old labels and old feature matrices are reused; no fit occurs. Explicit availability masks prevent missing telemetry from masquerading as zero motion.

In [ ]:
run_stage('smoke', 180)
smoke = read_result('features/smoke_summary.json')
assert smoke['status'] == 'motion_smoke_complete'
display(pd.DataFrame([{k:v for k,v in smoke.items() if k != 'training_only_support'}]))

In [ ]:
visuals.show_save(visuals.support_bars(smoke), HTML, '05_smoke_support')

## 2 · Extend the same features to the 256 existing plays
360-second cap. The 32 successful play checkpoints are reused. Raw output CSVs are not reloaded and no new examples are selected. This is not a new sample-size experiment.

In [ ]:
run_stage('prepare', 360)
prepared = read_result('features/preparation_summary.json')
assert prepared['status'] == 'motion_features_ready'
display(pd.DataFrame([{k:v for k,v in prepared.items() if k != 'training_only_support'}]))

In [ ]:
visuals.show_save(visuals.support_heatmap(prepared), HTML, '06_training_support')

## 3 · Measure each new family separately

**Arrival + turning (24)** versus saved arrival; **arrival + speed/braking (24)** versus saved arrival; **arrival + orientation discrepancy (12)** versus saved arrival. Each family includes explicit support channels and four-role gates. Maximum width is **120**, not 156. No combined arm is selected after viewing these results.

180-second cap; at most nine new fits. An addition earns further study only with >=1% pooled gain, adjusted upper difference bound below zero, and improvement in at least two folds. Negative results stop this tested interface from being scaled unchanged.

In [ ]:
run_stage('screen', 180)
motion = read_result('motion/summary.json')
assert motion['status'] == 'motion_complete'
display(pd.DataFrame(motion['decisions']))

In [ ]:
visuals.show_save(visuals.motion_scores(motion), HTML, '07_motion_scores')

In [ ]:
visuals.show_save(visuals.contrast_intervals(motion), HTML, '08_motion_intervals')

In [ ]:
visuals.show_save(visuals.horizon_scores(motion), HTML, '09_horizon_errors')

## 4 · Verify numerical restartability in a fresh process
180-second cap. This command is forbidden from fitting missing arms; it reloads and replays all 27 completed new models plus the six saved controls. Hash checks alone are not substituted for numerical predictions.

In [ ]:
run_stage('replay', 180)
for phase in ['attribution', 'motion']:
    replay = read_result(f'{phase}/replay_summary.json')
    assert replay['new_fits_this_invocation'] == 0
    assert replay['all_forward_replays_exact']
print('Fresh-process replay passed, with zero new fits.')

## 5 · Export the small return report

The ZIP contains aggregate results and receipts, not raw tracking, labels, per-row keys, feature matrices or weights. Save both notebooks and keep all private NPZ checkpoints on the persistent space. Download the report below and attach it in ChatGPT. Stop the JupyterLab space when finished; do not delete it.

**Next research decision:** integrate the justified arrival representation, plus only defensible companion families, into a separately frozen established-model comparison. Do not claim this diagnostic has beaten 0.46340, updated Kaggle, or exhausted the feature space.

In [ ]:
run_stage('report', 60)
print('DOWNLOAD THIS FILE:', OUT / 'nfl_feature_round2_report.zip')
print('Plotly HTML files:', HTML)
print('No GitHub push or model promotion occurred.')